# inplace-op-unsafe-warning — faded example 3: Restore the guard flag in finally

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-op-unsafe-warning`. Running the beacon reports progress on the `Backprop: In-place op unsafe warning` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: In-place op unsafe warning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-op-unsafe-warning`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-op-unsafe-warning"
DD_SUBTOPIC = "Backprop: In-place op unsafe warning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A context manager that temporarily disables the in-place guard must restore the prior flag value in a `finally` block. If the restore sits after the `yield` instead, an exception inside the `with` body skips it and leaves the guard disabled for the rest of the program.

## Faded exercise 3

Complete the `inplace_unsafe()` context manager. It saves the current `_GUARD_ARMED`, sets it `False` for the block, yields, and must re-arm it no matter what. Fill in the restore so it survives an exception in the body.

**Fill in:** the finally clause that restores the saved guard flag

In [ ]:
import contextlib

_GUARD_ARMED = True

@contextlib.contextmanager
def inplace_unsafe():
    global _GUARD_ARMED
    prev = _GUARD_ARMED
    _GUARD_ARMED = False
    try:
        yield
    finally:
        raise NotImplementedError()  # TODO: the finally clause that restores the saved guard flag


def _test():
    import builtins
    # normal block: disarmed inside, re-armed after
    assert _GUARD_ARMED is True
    with inplace_unsafe():
        assert _GUARD_ARMED is False
    assert _GUARD_ARMED is True
    # exception in body still re-arms via finally
    hit = False
    try:
        with inplace_unsafe():
            assert _GUARD_ARMED is False
            raise ValueError('boom')
    except ValueError:
        hit = True
    assert hit
    assert _GUARD_ARMED is True, 'guard must be re-armed even after an exception'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import contextlib

_GUARD_ARMED = True

@contextlib.contextmanager
def inplace_unsafe():
    global _GUARD_ARMED
    prev = _GUARD_ARMED
    _GUARD_ARMED = False
    try:
        yield
    finally:
        _GUARD_ARMED = prev
```
</details>